In [8]:
from pymilvus import connections

# Connect to Milvus
connections.connect(
    alias="default",
    uri="http://10.215.254.90:19530",
    db_name="chatbot_libra",
    user="chatbot_user",
    password="REPLACE_WITH_LOCAL_SECRET",
    pool_size=100,
    secure=False,
    timeout=10.0
)

print("Connected to Milvus successfully!")

Connected to Milvus successfully!


In [10]:
from pymilvus import Collection, utility

# Get the existing collection
collection = Collection("bot_semantic_search_10_12")

# Get schema
schema = collection.schema
print("Current schema:")
for field in schema.fields:
    print(f"Field: {field.name}, Type: {field.dtype}, Primary: {field.is_primary}")

# Get index information
index_info = collection.index().params
print("\nIndex information:")
print(index_info)

# Create new collection
new_collection_name = "bot_semantic_search_10_12_gwcode"

# Check if new collection exists and drop it
if utility.has_collection(new_collection_name):
    utility.drop_collection(new_collection_name)

# Create new collection with same schema
new_collection = Collection(name=new_collection_name, schema=schema)
print(f"\nCreated new collection: {new_collection_name}")

# Create index on new collection
field_name = collection.index().field_name  # Get the field name that was indexed in original collection
new_collection.create_index(field_name=field_name, index_params=index_info)
print(f"Created index on field '{field_name}' with params: {index_info}")

Current schema:
Field: role, Type: 21, Primary: False
Field: vector, Type: 101, Primary: False
Field: chunk_id, Type: 21, Primary: True
Field: topic, Type: 21, Primary: False
Field: type, Type: 21, Primary: False

Index information:
{'index_type': 'IVF_FLAT', 'metric_type': 'COSINE', 'params': {'nlist': 1024}}

Created new collection: bot_semantic_search_10_12_gwcode
Created index on field 'vector' with params: {'index_type': 'IVF_FLAT', 'metric_type': 'COSINE', 'params': {'nlist': 1024}}


In [17]:
from beegen.configs import EmbeddingConfig
from beegen.embeddings import TEIEmbedding

tei_embed = TEIEmbedding(EmbeddingConfig(
    api_type = "tei",
    api_key = "mb123456789",
    base_url = "http://10.1.39.127:30044/embed",
    model = "BAAI/bge-m3",
    embed_batch_size = 32,
    dimensions = 1024
))

embs = await tei_embed.run(texts = ['xin chào'])

In [1]:
from docx import Document

In [20]:
# Path to your file
file_path = r"D:\work\semantic_search\Tài liệu gửi team AI 11.12\NGÂN HÀNG ĐIỆN TỬ\18. Bảng mã lỗi APP MB BANK.docx"

# Read the document
doc = Document(file_path)

contents = []
# Read line by line from each paragraph
for paragraph in doc.paragraphs:
    text = paragraph.text.strip()  # Remove leading/trailing whitespace
    if text:  # Only process non-empty lines
        entries = text.split('\t')
        content = f"""Bảng mã lỗi APP MB BANK:\nCode: {entries[0]}\nResponse: {entries[1]}"""
        contents.append(content)

In [23]:
import json

# Save to JSON file
with open('output.json', 'w', encoding='utf-8') as f:
    json.dump(contents, f, ensure_ascii=False, indent=4)

In [25]:
import pandas as pd

# Convert to DataFrame with one column
df = pd.DataFrame(contents, columns=['content'])

# Save to CSV with UTF-8-SIG encoding
df.to_csv('output.csv', index=False, encoding='utf-8-sig')